# SenMortalityAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.SenMortalityAge)

class SenMortalityAge(LinearReferenceClock):
    pass



In [3]:
model = pya.models.SenMortalityAge()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "senmortalityage"
model.metadata["data_type"] = "DNA methylation"  # Paper: The predictors use CpG DNA-methylation beta values.
model.metadata["species"] = "Homo sapiens"  # Paper: All three assigned predictors were developed from human cell or human whole-blood datasets.
model.metadata["year"] = 2026
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Kasamoto, K., Gibson, J., Moqri, M., Smith, R. & Higgins-Chen, A.T. DNA methylation signatures of cellular senescence are not reversed by senolytic treatment. Aging Cell 25, e70430 (2026)."
model.metadata["doi"] = "https://doi.org/10.1111/acel.70430"
model.metadata["notes"] = "Senescence-enriched elastic-net Cox predictor of mortality, restricted to direction-concordant senescence/age/mortality CpGs and trained in the Framingham Heart Study."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: FHS whole-blood methylation was used for model training.
model.metadata["predicts"] = ["mortality risk"]  # Paper: SenMortalityAge is the mortality predictor among the three clocks.
model.metadata["training_target"] = ["mortality"]  # Paper: Elastic-net Cox regression was fitted to time-to-death.
model.metadata["unit"] = ["log hazard"]  # Paper: Pyaging returns the Cox linear predictor directly without exponentiation or another postprocess.
model.metadata["model_type"] = "elastic net Cox regression"  # Paper: The mortality model used elastic-net Cox regression with cross-validated lambda.
model.metadata["platform"] = ["Illumina 450K", "Illumina EPIC"]  # Paper: Feature discovery and predictor training used human 450K and EPIC methylation datasets.
model.metadata["population"] = "adults"  # Paper: The FHS cohorts comprised 2,748 Offspring and 1,457 Third Generation participants; the model used a 70/30 split.
model.metadata["journal"] = "Aging Cell"
model.metadata["last_author"] = "Albert T. Higgins-Chen"
model.metadata["n_features"] = 91
model.metadata["citations"] = 0
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o SenMortalityAge_CpGs.csv https://raw.githubusercontent.com/HigginsChenLab/methylCIPHER/19b12296b0d7eb7055a97d068064df635f44ce3e/data-raw/SenescenceAge/SenMortalityAge_CpGs.csv")

0

## Load features

In [6]:
coef_df = pd.read_csv('SenMortalityAge_CpGs.csv')
model.features = coef_df['CpG'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['Coefficient'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([0.0]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Kasamoto, Kotaro, et al. "DNA methylation clocks for estimating '
             'replicative senescence in human cells." Aging Cell (2026): '
             'e70430.',
 'clock_name': 'senmortalityage',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1111/acel.70430',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2026}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg00006787', 'cg00054496', 'cg00458878', 'cg00459119', 'cg03175417', 'cg03366574', 'cg04094193', 'cg04234014', 'cg05396212', 'cg06015525', 'cg06507987', 'cg06542681', 'cg07458308', 'cg08332990', 'cg08726900', 'cg08770961', 'cg10682299', 'cg11974796', 'cg12132563', 'cg12506165', 'cg13029847', 'cg13315970', 

## Normal feature ranges

Units and plausible bounds come from `pyaging`'s feature range registry, keyed by feature name with a fallback to the default for the clock's `data_type`. `predict_age` warns when input values fall outside these bounds, which usually means the data is in different units than the clock expects.

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: SenMortalityAge_CpGs.csv
